In [1]:

# TASK 1: DATABASE CREATION


import sqlite3
import pandas as pd

try:
    # Create Database Connection
    conn = sqlite3.connect("bookstore.db")
    cursor = conn.cursor()
    print("✓ Database connection established")

    # Enable Foreign Keys
    cursor.execute("PRAGMA foreign_keys = ON")

    # Create Books Table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        book_id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        author TEXT NOT NULL,
        price REAL NOT NULL,
        stock_quantity INTEGER DEFAULT 0
    )
    """)
    print("✓ Books table created successfully")

    # Create Customers Table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS customers (
        customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        city TEXT,
        join_date TEXT
    )
    """)
    print("✓ Customers table created successfully")

    # Create Orders Table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS orders (
        order_id INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_id INTEGER,
        book_id INTEGER,
        quantity INTEGER NOT NULL,
        order_date TEXT NOT NULL,
        total_amount REAL,
        FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
        FOREIGN KEY (book_id) REFERENCES books(book_id)
    )
    """)
    print("✓ Orders table created successfully")

    # Commit Table Creation
    conn.commit()

    # Display Schema using PRAGMA
    print("\nSchema for books table:")
    print(pd.read_sql("PRAGMA table_info(books);", conn))

    print("\nSchema for customers table:")
    print(pd.read_sql("PRAGMA table_info(customers);", conn))

    print("\nSchema for orders table:")
    print(pd.read_sql("PRAGMA table_info(orders);", conn))

except sqlite3.Error as e:
    print("Database error occurred:", e)


✓ Database connection established
✓ Books table created successfully
✓ Customers table created successfully
✓ Orders table created successfully

Schema for books table:
   cid            name     type  notnull dflt_value  pk
0    0         book_id  INTEGER        0       None   1
1    1           title     TEXT        1       None   0
2    2          author     TEXT        1       None   0
3    3           price     REAL        1       None   0
4    4  stock_quantity  INTEGER        0          0   0

Schema for customers table:
   cid         name     type  notnull dflt_value  pk
0    0  customer_id  INTEGER        0       None   1
1    1         name     TEXT        1       None   0
2    2        email     TEXT        1       None   0
3    3         city     TEXT        0       None   0
4    4    join_date     TEXT        0       None   0

Schema for orders table:
   cid          name     type  notnull dflt_value  pk
0    0      order_id  INTEGER        0       None   1
1    1   custo

In [2]:

# TASK 2: DATA INSERTION & QUERIES


try:

    # PART A: INSERT DATA

    # Books Data
    books_data = [
        ('Python Programming', 'John Smith', 599.99, 25),
        ('Data Science Handbook', 'Jane Doe', 899.50, 15),
        ('Machine Learning Basics', 'Alan Turing', 1299.00, 10),
        ('SQL Essentials', 'Edgar Codd', 499.99, 30),
        ('Web Development', 'Tim Berners', 799.00, 20)
    ]

    cursor.executemany("""
    INSERT INTO books (title, author, price, stock_quantity)
    VALUES (?, ?, ?, ?)
    """, books_data)
    print("✓ Inserted 5 books")


    # Customers Data
    customers_data = [
        ('Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15'),
        ('Priya Patel', 'priya@email.com', 'Delhi', '2024-01-20'),
        ('Amit Kumar', 'amit@email.com', 'Bangalore', '2024-02-01'),
        ('Sneha Reddy', 'sneha@email.com', 'Hyderabad', '2024-02-10'),
        ('Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')
    ]

    cursor.executemany("""
    INSERT INTO customers (name, email, city, join_date)
    VALUES (?, ?, ?, ?)
    """, customers_data)
    print("✓ Inserted 5 customers")


    # Orders Data
    orders_data = [
        (1, 1, 2, '2024-03-01', 1199.00),
        (1, 2, 1, '2024-03-02', 899.50),
        (2, 1, 1, '2024-03-03', 599.99),
        (2, 3, 1, '2024-03-05', 1299.00),
        (3, 4, 3, '2024-03-07', 1499.97),
        (4, 2, 1, '2024-03-10', 899.50),
        (5, 5, 2, '2024-03-12', 1598.00)
    ]

    cursor.executemany("""
    INSERT INTO orders (customer_id, book_id, quantity, order_date, total_amount)
    VALUES (?, ?, ?, ?, ?)
    """, orders_data)

    conn.commit()
    print("✓ Inserted 7 orders")


    # VERIFY INSERTION

    print("\nAll Books:")
    print(pd.read_sql("SELECT * FROM books", conn))

    print("\nAll Customers:")
    print(pd.read_sql("SELECT * FROM customers", conn))

    print("\nAll Orders:")
    print(pd.read_sql("SELECT * FROM orders", conn))


    # PART B: COMPLEX QUERIES


    print("\nCustomers from Mumbai:")
    print(pd.read_sql("""
    SELECT * FROM customers
    WHERE city = 'Mumbai'
    """, conn))


    print("\nBooks with price > 800 and stock > 10:")
    print(pd.read_sql("""
    SELECT * FROM books
    WHERE price > 800 AND stock_quantity > 10
    """, conn))


    print("\nTotal Number of Orders:")
    print(pd.read_sql("""
    SELECT COUNT(*) AS total_orders FROM orders
    """, conn))


    print("\nCustomer who placed the most orders:")
    print(pd.read_sql("""
    SELECT c.name, COUNT(o.order_id) AS order_count
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.name
    ORDER BY order_count DESC
    LIMIT 1
    """, conn))


    print("\nTotal Revenue:")
    print(pd.read_sql("""
    SELECT SUM(total_amount) AS total_revenue FROM orders
    """, conn))


except sqlite3.Error as e:
    print("Database error occurred:", e)

✓ Inserted 5 books
✓ Inserted 5 customers
✓ Inserted 7 orders

All Books:
   book_id                    title       author    price  stock_quantity
0        1       Python Programming   John Smith   599.99              25
1        2    Data Science Handbook     Jane Doe   899.50              15
2        3  Machine Learning Basics  Alan Turing  1299.00              10
3        4           SQL Essentials   Edgar Codd   499.99              30
4        5          Web Development  Tim Berners   799.00              20

All Customers:
   customer_id          name             email       city   join_date
0            1  Rahul Sharma   rahul@email.com     Mumbai  2024-01-15
1            2   Priya Patel   priya@email.com      Delhi  2024-01-20
2            3    Amit Kumar    amit@email.com  Bangalore  2024-02-01
3            4   Sneha Reddy   sneha@email.com  Hyderabad  2024-02-10
4            5  Vikram Singh  vikram@email.com     Mumbai  2024-02-15

All Orders:
   order_id  customer_id  book_id

In [3]:

# TASK 3: PANDAS INTEGRATION

try:

    # PART A: SQL → PANDAS


    # Load tables into DataFrames
    books_df = pd.read_sql("SELECT * FROM books", conn)
    customers_df = pd.read_sql("SELECT * FROM customers", conn)
    orders_df = pd.read_sql("SELECT * FROM orders", conn)

    print("Books DataFrame (First 3 rows):")
    print(books_df.head(3))

    print("\nCustomers DataFrame (First 3 rows):")
    print(customers_df.head(3))

    print("\nOrders DataFrame (First 3 rows):")
    print(orders_df.head(3))



    # Create Comprehensive Order Report

    report = orders_df.merge(customers_df, on="customer_id") \
                      .merge(books_df, on="book_id")

    final_report = report[['order_id','name','city','title','quantity','total_amount']]

    print("\nComprehensive Order Report:")
    print(final_report)



    # Analysis

    # Average Order Value
    avg_order_value = orders_df['total_amount'].mean()
    print("\nAverage Order Value:", round(avg_order_value, 2))

    # Total Orders by City
    orders_by_city = report.groupby("city")["order_id"].count()
    print("\nTotal Orders by City:")
    print(orders_by_city)

    # Most Popular Book (highest quantity sold)
    most_popular_book = report.groupby("title")["quantity"].sum().idxmax()
    print("\nMost Popular Book:", most_popular_book)


    # PART B: PANDAS → SQL


    discounts_data = {
        'book_id': [1, 2, 3, 4, 5],
        'discount_percent': [10, 15, 5, 20, 12]
    }

    discounts_df = pd.DataFrame(discounts_data)

    # Save to database
    discounts_df.to_sql("discounts", conn, if_exists="replace", index=False)

    print("\n✓ Discounts table created successfully")


    # Verify discounts table
    print("\nDiscounts Table:")
    print(pd.read_sql("SELECT * FROM discounts", conn))


    # Join Books with Discounts

    print("\nBooks with Discounted Prices:")
    print(pd.read_sql("""
    SELECT b.title,
           b.price AS original_price,
           d.discount_percent,
           ROUND(b.price - (b.price * d.discount_percent / 100.0), 2) AS discounted_price
    FROM books b
    JOIN discounts d ON b.book_id = d.book_id
    """, conn))


except Exception as e:
    print("Error occurred:", e)

Books DataFrame (First 3 rows):
   book_id                    title       author    price  stock_quantity
0        1       Python Programming   John Smith   599.99              25
1        2    Data Science Handbook     Jane Doe   899.50              15
2        3  Machine Learning Basics  Alan Turing  1299.00              10

Customers DataFrame (First 3 rows):
   customer_id          name            email       city   join_date
0            1  Rahul Sharma  rahul@email.com     Mumbai  2024-01-15
1            2   Priya Patel  priya@email.com      Delhi  2024-01-20
2            3    Amit Kumar   amit@email.com  Bangalore  2024-02-01

Orders DataFrame (First 3 rows):
   order_id  customer_id  book_id  quantity  order_date  total_amount
0         1            1        1         2  2024-03-01       1199.00
1         2            1        2         1  2024-03-02        899.50
2         3            2        1         1  2024-03-03        599.99

Comprehensive Order Report:
   order_id     

## Final Submission

Name: Lakshmi  
Database File: bookstore.db  
Notebook Link: [Paste your Drive/GitHub link here]  
Tasks Completed: Task 1 ✓, Task 2 ✓, Task 3 ✓  

### Key Insights:
- Designed relational database schema with primary and foreign keys.
- Implemented constraints like UNIQUE and NOT NULL.
- Performed SQL joins and aggregation queries.
- Integrated SQLite with Pandas for analysis.
- Used to_sql() to write DataFrame back to database.

Key Insights:
- Designed relational database schema with primary and foreign keys.
- Implemented constraints like UNIQUE and NOT NULL.
- Performed SQL joins and aggregation queries.
- Integrated SQLite with Pandas for analysis.
- Used to_sql() to write DataFrame back to database.